In [1]:
import json
import hashlib
import pandas as pd
from itertools import combinations

# --- Config ---
FILES = [
    "evals_1k.csv",
    "deepmind_test_62k.csv",
    "test_2k.csv",
    "train_20k.csv",
    "train_50k.csv",
]

In [2]:
# --- Load CSVs & gather FEN ID/FEN sets ---
dfs = {f: pd.read_csv(f, usecols=["FEN ID", "FEN"]) for f in FILES}
fen_id_sets = {f: set(df["FEN ID"]) for f, df in dfs.items()}
fen_sets = {f: set(df["FEN"]) for f, df in dfs.items()}

# --- In‑file FEN ID duplicates ---
print("✦ In‑file duplicate counts (by FEN ID)")
for f, df in dfs.items():
    dup_count = len(df) - df["FEN ID"].nunique()
    print(f"  {f:<22} -> {dup_count}")

# --- Pairwise overlaps ---
print("\n✦ Pairwise overlaps")
for (fa, fb) in combinations(FILES, 2):
    id_overlap = len(fen_id_sets[fa] & fen_id_sets[fb])
    fen_overlap = len(fen_sets[fa] & fen_sets[fb])
    print(f"  {fa:<22} ∩ {fb:<22} = FEN ID: {id_overlap:5d} | FEN: {fen_overlap:5d}")

✦ In‑file duplicate counts (by FEN ID)
  evals_1k.csv           -> 0
  deepmind_test_62k.csv  -> 1
  test_2k.csv            -> 0
  train_20k.csv          -> 0
  train_50k.csv          -> 0

✦ Pairwise overlaps
  evals_1k.csv           ∩ deepmind_test_62k.csv  = FEN ID:  1000 | FEN:  1000
  evals_1k.csv           ∩ test_2k.csv            = FEN ID:     0 | FEN:     0
  evals_1k.csv           ∩ train_20k.csv          = FEN ID:     0 | FEN:     0
  evals_1k.csv           ∩ train_50k.csv          = FEN ID:     0 | FEN:     0
  deepmind_test_62k.csv  ∩ test_2k.csv            = FEN ID:  2000 | FEN:  2000
  deepmind_test_62k.csv  ∩ train_20k.csv          = FEN ID: 20000 | FEN: 20000
  deepmind_test_62k.csv  ∩ train_50k.csv          = FEN ID: 50000 | FEN: 50000
  test_2k.csv            ∩ train_20k.csv          = FEN ID:     0 | FEN:     0
  test_2k.csv            ∩ train_50k.csv          = FEN ID:     0 | FEN:     0
  train_20k.csv          ∩ train_50k.csv          = FEN ID: 16812 | FEN: 16812


In [6]:
# Build global FEN→ID map (6 digit alphanumeric)
def fen_to_uid(fen):
    # Take sha256 hash, use first 6 hex digits, convert to base36 (alphanumeric), pad to 6
    h = hashlib.sha256(fen.strip().encode()).hexdigest()[:8]  # Take 8 to have more entropy
    as_int = int(h, 16)
    chars = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
    out = ''
    while as_int > 0 and len(out) < 6:
        out = chars[as_int % 36] + out
        as_int //= 36
    return out.rjust(6, '0')  # Pad to 6 chars

fen_to_id = {}
for fp in FILES:
    for fen in pd.read_csv(fp, usecols=["FEN"])["FEN"]:
        fen = str(fen).strip()
        if fen not in fen_to_id:
            fen_to_id[fen] = fen_to_uid(fen)

# Add “FEN ID” column and overwrite files
for fp in FILES:
    df = pd.read_csv(fp)
    df["FEN ID"] = df["FEN"].map(fen_to_id)
    df.to_csv(fp, index=False)

# Code to Generate FEN -> ID Mapping Once Already Generated   
---

In [6]:
# fen_id_mapping = dict()

# for f in FILES:
#     df = pd.read_csv(f, usecols=["FEN", "FEN ID"])
#     for fen, fen_id in zip(df["FEN"], df["FEN ID"]):
#         # If duplicate FENs exist, later files will overwrite earlier ones
#         fen_id_mapping[fen] = fen_id

# print(len(fen_id_mapping))

# with open("fen_id_mapping.json", "w") as f:
#     json.dump(fen_id_mapping, f, indent=2)

62561


In [ ]:
# import pandas as pd

# # Load all CSVs
# deepmind = pd.read_csv('deepmind_test_62k.csv')
# test = pd.read_csv('test_2k.csv')
# evals = pd.read_csv('evals_1k.csv')

# # Get set of FEN IDs to exclude
# exclude_ids = set(test['FEN ID']).union(set(evals['FEN ID']))

# # Filter deepmind to exclude overlaps
# filtered = deepmind[~deepmind['FEN ID'].isin(exclude_ids)]

# # Sample 50k without replacement
# train_50k = filtered.sample(n=50000, random_state=42)

# # Save to CSV
# train_50k.to_csv('train_50k.csv', index=False)